# 01 - Data Quality Profiling & Hygiene Audit
**Customer Retention & Churn Intelligence Platform**

This notebook performs:
1. Raw schema validation and column completeness
2. Missing values and blank string whitespace audit (notably `TotalCharges`)
3. Target class distribution and imbalance ratio
4. Numerical outlier analysis (IQR method) and categorical cardinalities


In [ ]:
import json
import pandas as pd
import numpy as np
from src.data.loader import load_raw_data
from src.validation.profiler import DataProfiler

# 1. Load Raw Dataset
df_raw = load_raw_data("data/raw/Telco-Customer-Churn.csv")
print(f"Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()


In [ ]:
# 2. Execute Data Profiler
profiler = DataProfiler(target_col="Churn", positive_val="Yes")
report = profiler.generate_quality_report(df_raw)

print("--- DATASET OVERVIEW ---")
print(json.dumps(report["dataset_overview"], indent=2))

print("\n--- TARGET DISTRIBUTION ---")
print(json.dumps(report["target_distribution"], indent=2))

print("\n--- WHITESPACE ANOMALIES (TotalCharges) ---")
print(json.dumps(report["missing_values"]["whitespace_anomalies"], indent=2))


### Finding & Treatment: TotalCharges Blanks
There are exactly 11 customer records with blank spaces `' '` in `TotalCharges`. All 11 records have `tenure = 0` (brand new subscribers). 
In our SQL staging pipeline (`sql/staging/02_stg_customers.sql`), we convert these to `0.0` with explicit documentation rather than dropping them.
